# Python with MySQL 기초
- 이 노트북은 `PyMySQL`을 사용해 MySQL에 연결하고 SQL을 직접 실행하는 실습입니다. 

> 가상환경 구축 
```shell
uv sync
```

## 1. 패키지 불러오기와 접속 정보 준비

상위 `docker-compose.yml`의 기본 접속 정보는 다음과 같습니다.

- host: `localhost`
- port: `3306`
- database: `examplesdb`
- user: `urstory`
- password: `u1234`

In [1]:
import os

import pandas as pd
import pymysql
from dotenv import load_dotenv

# .env 파일이 있으면 먼저 읽어옵니다.
# 없으면 아래 os.getenv()의 두 번째 값이 기본값으로 사용됩니다.
load_dotenv()

# MySQL 접속에 필요한 정보를 한 곳에 모아 둡니다.
# 이렇게 딕셔너리로 만들면 pymysql.connect(**DB_CONFIG)처럼 재사용하기 쉽습니다.
DB_CONFIG = {
    "host": os.getenv("DB_HOST", "localhost"),  # Docker MySQL을 로컬에서 접속할 때는 localhost를 사용합니다.
    "port": int(os.getenv("DB_PORT", "3306")),  # 포트는 숫자여야 하므로 int()로 변환합니다.
    "database": os.getenv("DB_NAME", "examplesdb"),
    "user": os.getenv("DB_USER", "urstory"),
    "password": os.getenv("DB_PASSWORD", "u1234"),
    "charset": "utf8mb4",  # 한글과 이모지까지 안전하게 저장하기 위한 문자셋입니다.
    "cursorclass": pymysql.cursors.DictCursor,  # 조회 결과를 튜플이 아니라 딕셔너리로 받습니다.
}

# 마지막 줄에 변수를 쓰면 Jupyter가 값을 화면에 보여줍니다.
DB_CONFIG

{'host': 'localhost',
 'port': 3306,
 'database': 'examplesdb',
 'user': 'urstory',
 'password': 'u1234',
 'charset': 'utf8mb4',
 'cursorclass': pymysql.cursors.DictCursor}

## 2. MySQL 연결 확인

`SELECT VERSION()`을 실행해서 MySQL 서버에 연결되는지 확인합니다.

In [2]:
# with 문을 사용하면 작업이 끝난 뒤 연결과 커서가 자동으로 정리됩니다.
with pymysql.connect(**DB_CONFIG) as conn:
    with conn.cursor() as cur:
        # execute()는 SQL 한 문장을 MySQL 서버로 보냅니다.
        cur.execute("SELECT VERSION() AS version;")
        # fetchone()은 결과 중 한 행만 가져옵니다.
        result = cur.fetchone()

result

{'version': '9.7.1'}

## 3. 실습 테이블 만들기

Python 실습 전용 테이블을 만듭니다. 같은 노트북을 여러 번 실행해도 되도록 `IF NOT EXISTS`를 사용합니다.

In [3]:
# 여러 줄 SQL은 따옴표 세 개로 작성하면 읽기 쉽습니다.
# IF NOT EXISTS를 붙이면 테이블이 이미 있어도 오류 없이 넘어갑니다.
create_table_sql = """
CREATE TABLE IF NOT EXISTS python_students (
    student_id INT AUTO_INCREMENT COMMENT '학생을 식별하는 자동 증가 기본키',
    name VARCHAR(50) NOT NULL COMMENT '학생의 이름',
    email VARCHAR(120) NOT NULL UNIQUE COMMENT '학생의 이메일, 중복 불가',
    score DECIMAL(5, 2) COMMENT 'Python 실습의 점수',
    created_at DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP COMMENT '데이터 등록 시간',
    PRIMARY KEY (student_id)
) COMMENT = 'Python 기초 실습용 학생 테이블';
"""

with pymysql.connect(**DB_CONFIG) as conn:
    with conn.cursor() as cur:
        cur.execute(create_table_sql)
    # CREATE/INSERT/UPDATE/DELETE처럼 DB 상태를 바꾸는 작업 뒤에는 commit()을 호출합니다.
    conn.commit()

print("python_students 테이블 준비 완료")

python_students 테이블 준비 완료


## 4. INSERT 실행하기

값을 SQL 문자열에 직접 붙이지 않고 `%s` 자리표시자와 파라미터를 사용합니다.

In [4]:
# INSERT할 데이터를 튜플 목록으로 준비합니다.
# 각 튜플의 순서는 INSERT문의 컬럼 순서(name, email, score)와 맞아야 합니다.
students = [
    ("김민준", "py_minjun@example.com", 92.5),
    ("이서연", "py_seoyeon@example.com", 85.0),
    ("최도윤", "py_doyun@example.com", 78.0),
]

insert_sql = """
INSERT INTO python_students (name, email, score)
VALUES (%s, %s, %s)
ON DUPLICATE KEY UPDATE
    name = VALUES(name),
    score = VALUES(score);
"""

with pymysql.connect(**DB_CONFIG) as conn:
    with conn.cursor() as cur:
        # executemany()는 같은 SQL을 여러 데이터에 반복 실행할 때 사용합니다.
        # %s 자리는 PyMySQL이 안전하게 값으로 바인딩합니다.
        cur.executemany(insert_sql, students)
    conn.commit()

print("데이터 입력 완료")

데이터 입력 완료


## 5. SELECT 결과 가져오기

`fetchall()`로 조회 결과를 가져온 뒤 pandas DataFrame으로 확인합니다.

In [5]:
# 조회할 컬럼과 정렬 기준을 SQL에 명확히 적어 둡니다.
select_sql = """
SELECT
    student_id,
    name,
    email,
    score,
    created_at
FROM python_students
ORDER BY student_id;
"""

with pymysql.connect(**DB_CONFIG) as conn:
    with conn.cursor() as cur:
        cur.execute(select_sql)
        # fetchall()은 조회된 모든 행을 리스트 형태로 가져옵니다.
        rows = cur.fetchall()

# 딕셔너리 목록을 DataFrame으로 바꾸면 노트북에서 표로 보기 좋습니다.
pd.DataFrame(rows)

,student_id,name,email,score,created_at
0,1,김민준,py_minjun@example.com,92.50,2026-06-29 01:21:42
1,2,이서연,py_seoyeon@example.com,85.00,2026-06-29 01:21:42
2,3,박도윤,py_doyun@example.com,78.00,2026-06-29 01:21:42


## 6. 조건을 사용해 조회하기

In [6]:
minimum_score = 80

with pymysql.connect(**DB_CONFIG) as conn:
    with conn.cursor() as cur:
        cur.execute(
            """
            SELECT name, email, score
            FROM python_students
            WHERE score >= %s
            ORDER BY score DESC;
            """,
            # 파라미터가 하나뿐이어도 튜플로 전달해야 하므로 쉼표를 붙입니다.
            (minimum_score,),
        )
        rows = cur.fetchall()

pd.DataFrame(rows)

,name,email,score
0,김민준,py_minjun@example.com,92.50
1,이서연,py_seoyeon@example.com,85.00


## 7. UPDATE와 DELETE 실행하기

In [7]:
with pymysql.connect(**DB_CONFIG) as conn:
    with conn.cursor() as cur:
        # UPDATE도 값을 직접 문자열로 합치지 않고 파라미터로 전달합니다.
        cur.execute(
            """
            UPDATE python_students
            SET score = %s
            WHERE email = %s;
            """,
            (95.0, "py_minjun@example.com"),
        )
        # 변경 결과를 바로 확인하기 위해 같은 연결에서 다시 SELECT합니다.
        cur.execute(
            """
            SELECT student_id, name, email, score
            FROM python_students
            WHERE email = %s;
            """,
            ("py_minjun@example.com",),
        )
        updated_row = cur.fetchone()
    conn.commit()

updated_row

{'student_id': 1,
 'name': '김민준',
 'email': 'py_minjun@example.com',
 'score': Decimal('95.00')}

In [8]:
with pymysql.connect(**DB_CONFIG) as conn:
    with conn.cursor() as cur:
        # 삭제하기 전에 어떤 행이 삭제될지 먼저 조회해 둡니다.
        cur.execute(
            """
            SELECT student_id, name, email
            FROM python_students
            WHERE email = %s;
            """,
            ("py_doyun@example.com",),
        )
        deleted_row = cur.fetchone()
        cur.execute(
            """
            DELETE FROM python_students
            WHERE email = %s;
            """,
            ("py_doyun@example.com",),
        )
    conn.commit()

# 삭제된 행 정보를 화면에 보여 주어 DELETE 결과를 확인합니다.
deleted_row

{'student_id': 3, 'name': '박도윤', 'email': 'py_doyun@example.com'}

## 8. mysql_db.py 연결 클래스 사용해보기

In [9]:
from mysql_db import MySQLDB

# MySQLDB는 mysql_db.py에 만든 간단한 연결 관리 클래스입니다.
# 같은 설정으로 여러 번 객체를 만들어도 내부에서는 하나의 연결을 재사용하도록 작성되어 있습니다.
db = MySQLDB(DB_CONFIG)

with db.get_conn().cursor() as cur:
    cur.execute("SELECT COUNT(*) AS student_count FROM python_students;")
    count_row = cur.fetchone()

count_row

{'student_count': 2}